## Ifyllning med MissForest och blockerad korsvalidering
*E. Haaf (Chalmers), 2026*

Denna notebook fyller luckor i prognosserier med MissForest.
Alla tillgängliga tidsserier används som prediktorer:
- SGU-referensserier
- Prognosserier


### 0. Importera bibliotek

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from missforest import MissForest
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### 1. Ladda in data

In [ ]:
# Konfiguration
sgu_path = "../data/sgu_data.csv"
prog_path = "../data/prog_gbg_long.csv"

filled_path = "../data/prognosis_filled_mf.csv"
metrics_path = "../data/cv_metrics_mf.csv"

min_obs_per_col = 10
start_date = pd.Timestamp("2018-01-01")

# MissForest-parametrar (kompatibla med missforest-paketet)
mf_max_iter = 5
mf_early_stopping = True
mf_verbose = 1

In [ ]:
def _read_long_csv(path):
    df = pd.read_csv(path, sep=";", na_values=["NA"])
    df.columns = [c.strip() for c in df.columns]
    if not {"Date", "ID", "Value"}.issubset(df.columns):
        raise ValueError(f"Forväntade kolumner Date, ID, Value i {path}")
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Value"] = (
        df["Value"]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .replace({"NA": np.nan, "nan": np.nan})
    )
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    return df.dropna(subset=["Date"])


def _long_to_wide(df_long, prefix):
    wide = df_long.copy()
    wide["ID"] = prefix + wide["ID"].astype(str)
    wide = (
        wide.groupby(["Date", "ID"], as_index=False)["Value"]
        .mean()
        .pivot(index="Date", columns="ID", values="Value")
        .sort_index()
    )
    return wide


def _fit_transform_missforest(df_in, verbose_level=0):
    """Kör MissForest och returnerar alltid en DataFrame."""
    mf = MissForest(
        max_iter=mf_max_iter,
        early_stopping=mf_early_stopping,
        verbose=verbose_level,
    )
    out = mf.fit_transform(df_in)
    if isinstance(out, pd.DataFrame):
        return out
    return pd.DataFrame(out, index=df_in.index, columns=df_in.columns)


sgu_long = _read_long_csv(sgu_path)
prog_long = _read_long_csv(prog_path)

# Filtrera bort data före start_date
sgu_long = sgu_long[sgu_long["Date"] >= start_date].copy()
prog_long = prog_long[prog_long["Date"] >= start_date].copy()

sgu_wide = _long_to_wide(sgu_long, "sgu_")
prog_wide = _long_to_wide(prog_long, "prog_")

print(f"SGU kolumner (referens): {sgu_wide.shape[1]}")
print(f"Prognos kolumner: {prog_wide.shape[1]}")

### 2. Bygg prognosmatris

In [ ]:
# Bygg gemensam feature-matris
X = pd.concat([prog_wide, sgu_wide], axis=1).sort_index()

# Ta bort kolumner utan tillräckligt med observationer
valid_cols = X.columns[X.notna().sum() >= min_obs_per_col]
X = X[valid_cols]

prog_cols = [c for c in X.columns if c.startswith("prog_")]
if not prog_cols:
    raise ValueError("Inga prognoskolumner hittades efter filtrering")

print(f"Matrisstorlek: {X.shape[0]} datum x {X.shape[1]} kolumner")
print(f"Antal prognoskolumner: {len(prog_cols)}")
X.head()

SGU kolumner (referens): 8
Prognos kolumner: 9


### 3. Fyll luckor och kör korsvalidering

In [ ]:
# MissForest-imputering på hela matrisen
t0 = time.time()
X_filled = _fit_transform_missforest(X, verbose_level=mf_verbose)
elapsed = time.time() - t0

prog_raw = X[prog_cols]
prog_filled = X_filled[prog_cols]

print(f"Imputering klar ({elapsed:.1f} s)")

In [ ]:
# Blockerad korsvalidering (2-månadersblock)
cv_base = X.copy()
folds = cv_base.index.to_series().dt.to_period("2M")

metrics_rows = []
for col in prog_cols:
    y_true_all = []
    y_pred_all = []

    for fold in folds.dropna().unique():
        test_mask = folds == fold
        observed_mask = cv_base[col].notna() & test_mask
        if observed_mask.sum() == 0:
            continue

        X_fold = cv_base.copy()
        X_fold.loc[observed_mask, col] = np.nan

        X_fold_filled = _fit_transform_missforest(X_fold, verbose_level=0)

        y_true_all.append(cv_base.loc[observed_mask, col].to_numpy())
        y_pred_all.append(X_fold_filled.loc[observed_mask, col].to_numpy())

    if y_true_all:
        y_true = np.concatenate(y_true_all)
        y_pred = np.concatenate(y_pred_all)
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
        n_eval = len(y_true)
    else:
        mae = np.nan
        rmse = np.nan
        r2 = np.nan
        n_eval = 0

    metrics_rows.append({
        "ID": col.replace("prog_", ""),
        "column": col,
        "n_eval": n_eval,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
    })

metrics_df = pd.DataFrame(metrics_rows).sort_values("ID")
print("CV klar")
display(metrics_df.head(20))

Matrisstorlek: 2649 datum x 17 kolumner
Antal prognoskolumner: 9


ID,prog_obj_06,prog_obj_07,prog_obj_2U,prog_obj_30,prog_obj_3O,prog_obj_5B,prog_obj_5U,prog_obj_8O,prog_obj_9O,sgu_179_2,sgu_179_3,sgu_52_11,sgu_52_13,sgu_52_14,sgu_52_2,sgu_52_8,sgu_52_9
Date,,,,,,,,,,,,,,,,,
2018-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.475667,46.600333,59.781333
2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.150496,NaN,NaN,NaN,NaN,NaN,NaN,52.455333,46.595000,59.780833
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.142406,NaN,NaN,NaN,NaN,NaN,NaN,52.455667,46.593333,59.783667
2018-01-04,NaN,NaN,14.418,NaN,14.214,NaN,NaN,24.140746,NaN,NaN,NaN,NaN,NaN,NaN,52.404167,46.552000,59.783750
2018-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.148246,18.73,NaN,NaN,NaN,NaN,NaN,52.361667,46.513667,NaN


### 4. Spara och visa resultat

In [ ]:
# Spara resultat
filled_long = (
    prog_filled.reset_index()
    .melt(id_vars="Date", var_name="column", value_name="Value")
)
filled_long["ID"] = filled_long["column"].str.replace("^prog_", "", regex=True)

raw_long = (
    prog_raw.reset_index()
    .melt(id_vars="Date", var_name="column", value_name="Value_raw")
)

filled_export = filled_long.merge(raw_long, on=["Date", "column"], how="left")
filled_export = filled_export[["Date", "ID", "Value", "Value_raw", "column"]]

filled_export.to_csv(filled_path, sep=";", index=False)
metrics_df.to_csv(metrics_path, sep=";", index=False)

print(f"Sparad fylld prognos: {filled_path}")
print(f"Sparade metriker: {metrics_path}")

### 5. Enkel visualisering

In [ ]:
# Minimal visualisering
ids_to_plot = filled_export["ID"].drop_duplicates().head(3).tolist()
if ids_to_plot:
    fig, axes = plt.subplots(len(ids_to_plot), 1, figsize=(12, 4 * len(ids_to_plot)), sharex=True)
    if len(ids_to_plot) == 1:
        axes = [axes]

    for ax, id_ in zip(axes, ids_to_plot):
        part = filled_export[filled_export["ID"] == id_].sort_values("Date")
        ax.plot(part["Date"], part["Value_raw"], color="black", linewidth=1.2, label="Observerad")
        ax.plot(part["Date"], part["Value"], color="tab:blue", linewidth=1.0, label="MissForest")
        ax.set_title(f"Prognos ID {id_}")
        ax.grid(True, alpha=0.3)
        ax.legend()

    axes[-1].set_xlabel("Datum")
    plt.tight_layout()
    plt.show()